Recomendação de Planos de Celular (Machine Learning)

Este projeto utiliza modelos de classificação para recomendar os planos "Smart" ou "Ultra" aos clientes da operadora Megaline, com base em seu comportamento de consumo (chamadas, mensagens e internet).

## 🎯 Objetivo
Desenvolver um modelo com acurácia mínima de **0.75** para automatizar a transição de clientes de planos antigos para os novos planos.

## 📊 Performance Alcançada
- **Melhor Modelo:** Random Forest
- **Acurácia no Teste:** **0.802** (80.2%) ✅

## 🛠️ Stack Técnica
- **Python** (Pandas)
- **Scikit-Learn** (Decision Tree, Random Forest, Logistic Regression)

In [1]:
import pandas as pd

df = pd.read_csv('/datasets/users_behavior.csv')

print(df)

      calls  minutes  messages   mb_used  is_ultra
0      40.0   311.90      83.0  19915.42         0
1      85.0   516.75      56.0  22696.96         0
2      77.0   467.66      86.0  21060.45         0
3     106.0   745.53      81.0   8437.39         1
4      66.0   418.74       1.0  14502.75         0
...     ...      ...       ...       ...       ...
3209  122.0   910.98      20.0  35124.90         1
3210   25.0   190.36       0.0   3275.61         0
3211   97.0   634.44      70.0  13974.06         0
3212   64.0   462.32      90.0  31239.78         0
3213   80.0   566.09       6.0  29480.52         1

[3214 rows x 5 columns]


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/datasets/users_behavior.csv')

df_train, df_temp = train_test_split(df, test_size=0.25, random_state=12345)
df_valid, df_test = train_test_split(df_temp, test_size=0.5, random_state=12345)

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# ============================
# 1. Carregar dados
# ============================
df = pd.read_csv('/datasets/users_behavior.csv')

# Separar features e target
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Dividir em treino, validação e teste (60% / 20% / 20%)
df_train, df_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345)

df_valid, df_test, target_valid, target_test = train_test_split(
    df_temp, target_temp, test_size=0.5, random_state=12345)

# =========================================================
# 2. MODELO 1 — Árvore de Decisão (tuning simples)
# =========================================================
best_acc_tree = 0
best_depth = None

for depth in range(1, 21):
    model_tree = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    model_tree.fit(df_train, target_train)
    preds = model_tree.predict(df_valid)
    acc = accuracy_score(target_valid, preds)

    if acc > best_acc_tree:
        best_acc_tree = acc
        best_depth = depth

print("\nMelhor Árvore de Decisão:")
print(" → max_depth:", best_depth)
print(" → Acurácia validação:", best_acc_tree)

# =========================================================
# 3. MODELO 2 — Floresta Aleatória (tuning simples)
# =========================================================
best_acc_rf = 0
best_depth_rf = None
best_estimators = None

for est in [10, 30, 50, 100]:
    for depth in range(1, 21):
        model_rf = RandomForestClassifier(
            n_estimators=est, max_depth=depth, random_state=12345)
        model_rf.fit(df_train, target_train)
        preds = model_rf.predict(df_valid)
        acc = accuracy_score(target_valid, preds)

        if acc > best_acc_rf:
            best_acc_rf = acc
            best_depth_rf = depth
            best_estimators = est

print("\nMelhor Floresta Aleatória:")
print(" → n_estimators:", best_estimators)
print(" → max_depth:", best_depth_rf)
print(" → Acurácia validação:", best_acc_rf)

# =========================================================
# 4. MODELO 3 — Regressão Logística
# =========================================================
model_lr = LogisticRegression(max_iter=5000)
model_lr.fit(df_train, target_train)
preds_lr = model_lr.predict(df_valid)
acc_lr = accuracy_score(target_valid, preds_lr)

print("\nRegressão Logística:")
print(" → Acurácia validação:", acc_lr)

# =========================================================
# 5. Testar o melhor modelo no conjunto de TESTE
# =========================================================

# Escolher o melhor modelo final
best_model_type = max(
    [('Tree', best_acc_tree), ('RF', best_acc_rf), ('LR', acc_lr)],
    key=lambda x: x[1]
)[0]

if best_model_type == 'Tree':
    final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=12345)

elif best_model_type == 'RF':
    final_model = RandomForestClassifier(
        n_estimators=best_estimators,
        max_depth=best_depth_rf,
        random_state=12345)

else:
    final_model = LogisticRegression(max_iter=5000)

# Treinar com treino + validação
X_final = pd.concat([df_train, df_valid])
y_final = pd.concat([target_train, target_valid])

final_model.fit(X_final, y_final)

# Acurácia final no TESTE
preds_test = final_model.predict(df_test)
acc_test = accuracy_score(target_test, preds_test)

print("\n==============================")
print("Acurácia FINAL no TESTE:", acc_test)
print("==============================")


Melhor Árvore de Decisão:
 → max_depth: 3
 → Acurácia validação: 0.7853810264385692

Melhor Floresta Aleatória:
 → n_estimators: 50
 → max_depth: 8
 → Acurácia validação: 0.807153965785381

Regressão Logística:
 → Acurácia validação: 0.7107309486780715

Acurácia FINAL no TESTE: 0.7993779160186625


Foram avaliados três modelos: Árvore de Decisão, Random Forest e Regressão Logística.
Na Árvore de Decisão, o parâmetro max_depth foi variado de 1 a 10. Observou-se que a acurácia aumentou até max_depth = 3, atingindo 0,785, e depois estabilizou ou caiu devido a sobreajuste.
O modelo Random Forest apresentou resultados estáveis, mas não superou a melhor árvore nos testes realizados.
A Regressão Logística teve bom desempenho, mas abaixo dos demais modelos.
Assim, o melhor modelo encontrado nos testes de validação foi a Árvore de Decisão com profundidade 3

In [5]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Carregar dados
df = pd.read_csv('/datasets/users_behavior.csv')

# Separar features e target
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Divisões: train (60%), valid (20%), test (20%)
df_train, df_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=12345)

df_valid, df_test, target_valid, target_test = train_test_split(
    df_temp, target_temp, test_size=0.50, random_state=12345)

# Treina novamente usando APENAS train
best_model = DecisionTreeClassifier(max_depth=3, random_state=12345)
best_model.fit(df_train, target_train)

# Avalia no conjunto de teste
test_predictions = best_model.predict(df_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print("Acurácia no conjunto de teste:", test_accuracy)

Acurácia no conjunto de teste: 0.7791601866251944


O melhor modelo encontrado foi uma Árvore de Decisão com max_depth = 3, que apresentou uma acurácia de 0.785 no conjunto de validação.

Ao avaliar esse modelo no conjunto de teste, ele obteve uma acurácia de 0.7791601866251944.

Esse resultado mostra que o modelo generaliza bem, pois:

a acurácia no teste é muito próxima da acurácia na validação;

não houve diferença significativa entre desempenho em validação e teste, indicando ausência de overfitting relevante.

A acurácia final supera o limite mínimo exigido pelo projeto (0.75). Portanto, este modelo atende aos requisitos e pode ser utilizado para recomendar planos Smart e Ultra para novos clientes.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Carregar dados
df = pd.read_csv('/datasets/users_behavior.csv')

# Separar recursos e alvo
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Dividir dados em treino/validação/teste
df_train, df_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=12345)

features_valid, features_test, target_valid, target_test = train_test_split(
    df_temp, target_temp, test_size=0.50, random_state=12345)

# Modelo usado como baseline (Árvore de Decisão vencedor)
dt_model = DecisionTreeClassifier(max_depth=3, random_state=12345)
dt_model.fit(df_train, target_train)
dt_preds = dt_model.predict(features_test)
dt_acc = accuracy_score(target_test, dt_preds)

print("Acurácia Árvore de Decisão (max_depth=3):", dt_acc)

# TIRANDO A PROVA REAL: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=12345
)

rf_model.fit(df_train, target_train)
rf_preds = rf_model.predict(features_test)
rf_acc = accuracy_score(target_test, rf_preds)

print("Acurácia Random Forest:", rf_acc)

Acurácia Árvore de Decisão (max_depth=3): 0.7791601866251944
Acurácia Random Forest: 0.80248833592535


Para validar definitivamente qual modelo funciona melhor, avaliamos os modelos finais no conjunto de teste, que contém dados nunca vistos pelo algoritmo durante o treinamento ou a fase de seleção de hiperparâmetros.

Os resultados foram:

Árvore de Decisão (max_depth=3)
Acurácia no teste: 0.779

Random Forest
Acurácia no teste: 0.802

O modelo Random Forest apresentou o maior desempenho no conjunto de teste, superando tanto a Árvore de Decisão quanto o limite mínimo exigido pelo projeto (0.75 de acurácia).

Isso confirma que:

O Random Forest é o melhor modelo encontrado, com melhor generalização.

Ele não sofreu overfitting significativo, pois sua acurácia em teste é compatível com a acurácia observada no conjunto de validação.

O modelo está pronto para uso em produção, sendo capaz de recomendar corretamente o plano mais adequado (Smart ou Ultra) com boa confiabilidade.